In [1]:
import pandas as pd
import numpy as np
import math
import os
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
os.getcwd()
os.chdir(os.getcwd())
os.getcwd()

'/home/unimelb.edu.au/gargerd/data/C-PATH/outcome_prediction/outcome_prediction'

In [4]:
vs=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/vs.csv',low_memory=False)

### Check how many patients have height data

In [5]:
patients_for_analyis=pd.read_csv('../data/patients_in_analysis.csv.gz')

a=vs.loc[vs['VSTESTCD'].isin(['HEIGHT']),['STUDYID','USUBJID','VSTEST','VSSTRESN','VSSTRESU','VISIT','EPOCH','VSDY']].dropna(how='all',axis=1)
print('total number of patients with height data: ',len((a['USUBJID'].unique())))
print('total number of patients for TTP pred analyis: ',len((patients_for_analyis['USUBJID'].unique())))
print('total number of patients with height info in TTP pred. analysis: ',len(set(a['USUBJID'])&set(patients_for_analyis['USUBJID'])))
pats_wo_height=set(patients_for_analyis['USUBJID'])-set(set(a['USUBJID'])&set(patients_for_analyis['USUBJID']))
vs_wo_height=vs[vs['USUBJID'].isin(pats_wo_height)]

print('\n\npatients without height data')
print(vs_wo_height['STUDYID'].value_counts())

total number of patients with height data:  15997
total number of patients for TTP pred analyis:  5796
total number of patients with height info in TTP pred. analysis:  3612


patients without height data
STUDYID
TB-1030    1352
TB-1020     827
TB-1018      96
TB-1021      56
Name: count, dtype: int64


## Create scaled values and subset the dataframe to patients considered in analysis

In [20]:
pat_ids=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)

vs['STD_NUM_RESULT_SCALED']=np.nan
for test,df in vs.groupby(by=['VSTEST']):
    df['VSTEST']
    # Create scaled Z-score
    vs.loc[df.index,'STD_NUM_RESULT_SCALED']=StandardScaler().fit_transform(df['VSSTRESN'].values.reshape(-1,1))

## Standardise Heart Rate name
vs['STD_VSTEST']=vs['VSTEST']
vs.loc[vs['VSTEST']=='Pulse Rate','STD_VSTEST']='Heart Rate'

vs_std=vs[vs['USUBJID'].isin(pat_ids['USUBJID'].unique().tolist())]

/home/unimelb.edu.au/gargerd/anaconda3/envs/xenium/lib/python3.10/site-packages/sklearn/utils/extmath.py:1050: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/home/unimelb.edu.au/gargerd/anaconda3/envs/xenium/lib/python3.10/site-packages/sklearn/utils/extmath.py:1055: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/home/unimelb.edu.au/gargerd/anaconda3/envs/xenium/lib/python3.10/site-packages/sklearn/utils/extmath.py:1075: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


## Save standardised dataframe

In [22]:
vs_std.to_csv('../data/out_vs_standardised.csv.gz',compression='gzip')

### Create dataframe with available wright-height-BMI data, as it is necessary for PK modeling

In [23]:
colnames=['STUDYID','USUBJID','VSTEST','VSSTRESN','VSSTRESU','VISIT','EPOCH','VSDY','STD_VSTEST','STD_NUM_RESULT_SCALED']
weight_height_df=vs.loc[(vs['VSTESTCD'].isin(['WEIGHT','HEIGHT','BMI']))&(vs['USUBJID'].isin(patients_for_analyis['USUBJID'].unique())),colnames]
weight_height_df=weight_height_df.sort_values(by=['USUBJID','VSDY'])
weight_height_df.to_csv('../data/weight_height_bmi_of_patients.csv.gz',compression='gzip')